微调：迁移学习的一种技术，即使用预训练神经网络的权重作为初始值，在不同任务上训练新的神经网络。

迁移学习：可重复使用现有模型，而非从头开始训练新模型，从而节省时间和资源。

特征提取：在基础模型上训练分类器层的过程。

迁移学习和微调的区别：迁移学习是在基础模型上添加一个新的分类器层，然后只训练该层。微调则是解冻部分或全部基础模型层，然后与分类器层一起训练。

微调步骤：

1.加载预训练的语言模型及其词元分析器。

2.准备特定任务数据集：数据集应包含与任务相关的输入-输出对。

3.定义特定任务的头：“头”指的是在预训练模型之上添加的一个层或一组层，用于执行任务。

4.在特定任务数据集上训练模型。

5.在测试集或验证集上评估模型。

In [2]:
from datasets import load_dataset

D:\Anaconda\envs\openai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [4]:
dataset = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [5]:
#使用切片器访问特定数据集对象具有的某个观察值（例如train）
dataset["train"][100]

{'text': "Terrible movie. Nuff Said.<br /><br />These Lines are Just Filler. The movie was bad. Why I have to expand on that I don't know. This is already a waste of my time. I just wanted to warn others. Avoid this movie. The acting sucks and the writing is just moronic. Bad in every way. The only nice thing about the movie are Deniz Akkaya's breasts. Even that was ruined though by a terrible and unneeded rape scene. The movie is a poorly contrived and totally unbelievable piece of garbage.<br /><br />OK now I am just going to rag on IMDb for this stupid rule of 10 lines of text minimum. First I waste my time watching this offal. Then feeling compelled to warn others I create an account with IMDb only to discover that I have to write a friggen essay on the film just to express how bad I think it is. Totally unnecessary.",
 'label': 0}

In [6]:
#初始化词元分析器
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
def tokenize_function(examples):
    return tokenizer(examples["text"],padding = "max_length", truncation=True)
tokenized_datasets = dataset.map(tokenize_function, batched=True)

上述代码配置了tokenize_function的填充和截断字段，以确保输出的大小适合BERT模型

In [7]:
tokenized_datasets['train'][100]['input_ids']

[101,
 12008,
 27788,
 2523,
 119,
 151,
 9435,
 11455,
 119,
 133,
 9304,
 120,
 135,
 133,
 9304,
 120,
 135,
 1636,
 12058,
 1132,
 2066,
 17355,
 9860,
 119,
 1109,
 2523,
 1108,
 2213,
 119,
 2009,
 146,
 1138,
 1106,
 7380,
 1113,
 1115,
 146,
 1274,
 112,
 189,
 1221,
 119,
 1188,
 1110,
 1640,
 170,
 5671,
 1104,
 1139,
 1159,
 119,
 146,
 1198,
 1458,
 1106,
 11857,
 1639,
 119,
 138,
 6005,
 2386,
 1142,
 2523,
 119,
 1109,
 3176,
 22797,
 1105,
 1103,
 2269,
 1110,
 1198,
 182,
 14824,
 7770,
 119,
 6304,
 1107,
 1451,
 1236,
 119,
 1109,
 1178,
 3505,
 1645,
 1164,
 1103,
 2523,
 1132,
 14760,
 9368,
 138,
 19610,
 2315,
 112,
 188,
 13016,
 119,
 2431,
 1115,
 1108,
 9832,
 1463,
 1118,
 170,
 6434,
 1105,
 8362,
 23063,
 4902,
 9372,
 2741,
 119,
 1109,
 2523,
 1110,
 170,
 9874,
 14255,
 19091,
 5790,
 1105,
 5733,
 8362,
 26438,
 2727,
 1104,
 14946,
 119,
 133,
 9304,
 120,
 135,
 133,
 9304,
 120,
 135,
 10899,
 1208,
 146,
 1821,
 1198,
 1280,
 1106,
 26133,
 1113,
 

In [8]:
#缩短训练时间，可以选择缩小数据集的大小
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(500))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(500))

微调模型：

BERT模型主要组成部分：编码器、输出层；其中模型的层数和参数取决于模型版本

模型训练阶段目标：

1.掩码语言建模（MLM）：教会模型预测输入文本中被随机掩码（用特殊词元替换）的原词。

2.下一句预测（NSP）：教会模型预测原文中的两个句子是否连续。

In [9]:
import torch
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased",num_labels=2)

Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 1490.47it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized

LLM常用评估指标：

流畅性、连贯性、相关性、GPT相似性、基础性

评估二元分类器的基本方法之一——使用混淆矩阵

1.真阳性（TP）：当真实标签为1，分类器正确预测为1的案例数

2.假阳性（FP）：当真实标签为0，分类器错误预测为1的案例数

3.真阴性（TN）：当真实标签为0，分类器正确预测为0的案例数

4.假阴性（FN）：当真实标签为1，分类器错误预测为0的案例数

混淆矩阵中常见的指标使用：

1.准确性：所有预测（TP+TN）/（TP+FP+TN+FN）

2.查准率：阳性预测中，正确预测所占比例，（TP）/（TP+FP）

3.召回率：在所有阳性案例中，真阳性预测的数量所占的比例，也称灵敏度或真阳性率。（TP）/（TP+FN）

4.特异性：在所有阴性案例中，真阴性预测所占比例，（TN）/（TN+FP）

5.FI分数：查准率和召回率的调和平均值。它是查准率和召回率之间的平衡度量。公式：2*（查准率*召回率）/（查准率+召回率）

In [10]:
import numpy as np
import evaluate
metric = evaluate.load("accuracy")

定义函数，用于计算训练阶段输出的准确性

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [12]:
#设置评估策略，即希望模型在训练过程中针对测试集进行测试的频率
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="D:/test_trainer", 
    num_train_epochs=10,   
    eval_strategy="epoch",
    save_strategy="epoch",          
    load_best_model_at_end=True,    
    metric_for_best_model="accuracy", 
    save_total_limit=2,             
)

迭代周期：epoch，用来描述对整个训练数据集的一次完整遍历，是一个超参数，可通过微调来提高机器学习模型的性能。

Trainer对象：
需要用到一个模型、一些配置参数（如迭代周期数量）、一个训练数据集、一个评估数据集以及要计算的评估指标类型

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

In [14]:
trainer.train()

D:\Anaconda\envs\openai_env\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.561018,0.770000
2,No log,0.626826,0.840000
3,No log,0.660052,0.868000
4,No log,0.656049,0.874000
5,No log,0.751612,0.868000


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.11s/it]
D:\Anaconda\envs\openai_env\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.18s/it]
D:\Anaconda\envs\openai_env\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.13s/it]
D:\Anaconda\envs\openai_env\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned me

KeyboardInterrupt: 

In [15]:
# 查看当前最佳结果
logs = trainer.state.log_history
for log in logs:
    if 'eval_accuracy' in log:
        print(f"Epoch {log.get('epoch', '?'):.0f}: Acc={log['eval_accuracy']:.4f}, Loss={log.get('eval_loss', '?'):.4f}")

Epoch 1: Acc=0.7700, Loss=0.5610
Epoch 2: Acc=0.8400, Loss=0.6268
Epoch 3: Acc=0.8680, Loss=0.6601
Epoch 4: Acc=0.8740, Loss=0.6560
Epoch 5: Acc=0.8680, Loss=0.7516


In [16]:
trainer.save_model('models/sentiment-classifier')

Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.95s/it]


In [18]:
#使用和测试模型
model = AutoModelForSequenceClassification.from_pretrained('models/sentiment-classifier')

inputs = tokenizer("I cannot stand it anymore!", return_tensors="pt")
outputs = model(**inputs)
outputs

Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 1308.71it/s]


SequenceClassifierOutput(loss=None, logits=tensor([[ 1.6176, -1.7340]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [23]:
#reducing the rows of the dataset to leverage the free autotrain

import pandas as pd

small_dataset = dataset["train"].shuffle(seed=42).select(range(3000))
small_dataset = small_dataset.to_pandas()

small_dataset['label'] = small_dataset['label'].replace({0: 'Negative', 1: 'Positive'})

small_dataset.head()

small_dataset.to_csv('imdb_small.csv')

In [25]:
import tensorflow as tf


predictions = tf.math.softmax(outputs.logits.detach(), axis=-1)
print(predictions)

tf.Tensor([[0.9661599  0.03384012]], shape=(1, 2), dtype=float32)


In [26]:
model.config.id2label

{0: 'LABEL_0', 1: 'LABEL_1'}

In [27]:
from transformers import pipeline

classifier = pipeline(task ='sentiment-analysis', model = model, tokenizer = tokenizer)
classifier('I cannot believe you did it again!')

[{'label': 'LABEL_0', 'score': 0.7872326374053955}]

In [28]:
#运用TensorFlow库来显示张量，使用softmax函数来获取与每个标签相关的概率向量，得到最终结果出现概率最大的标签
import tensorflow as tf
predictions = tf.math.softmax(outputs.logits.detach(), axis=-1)
print(predictions)

tf.Tensor([[0.9661599  0.03384012]], shape=(1, 2), dtype=float32)
